# Process model widgets

Each widget shows one level of a CADET-Process model and edits the object it is given: components, unit operations, flow sheet, process, simulation.

Here, every value is also set in code, so the notebook runs without clicking. **Show code** in each widget shows the CADET-Process code of that step.

## Empty model objects

The widgets fill these objects.

In [ ]:
from CADETProcess.processModel import (
    ComponentSystem, FlowSheet, Inlet, LumpedRateModelWithoutPores, Outlet, Process,
)

component_system = ComponentSystem()
units = {}
flow_sheet = FlowSheet(component_system, "flow_sheet")
process = Process(flow_sheet, "tracer_pulse")

## 1. Components

In [ ]:
from cadetgui import ComponentSystemWidget

component_system.add_component("Tracer")
ComponentSystemWidget(component_system)

## 2. Unit operations

An inlet, a lumped rate model column and an outlet. Pick a unit under **Parameters of** to edit its parameters.

In [ ]:
from cadetgui import UnitOperationsWidget

inlet = Inlet(component_system, "inlet")
inlet.flow_rate = 1e-7

column = LumpedRateModelWithoutPores(component_system, "column")
column.length = 0.1
column.diameter = 0.02
column.total_porosity = 0.4
column.axial_dispersion = 1e-7

outlet = Outlet(component_system, "outlet")

units.update(inlet=inlet, column=column, outlet=outlet)
UnitOperationsWidget(component_system, units)

## 3. Flow sheet

`connections` adds the units and connects them.

In [ ]:
from cadetgui import FlowSheetWidget

FlowSheetWidget(flow_sheet, units=units, connections=[("inlet", "column"), ("column", "outlet")])

## 4. Process

A pulse: the inlet feeds the tracer for 60 s, then only buffer. The events are drawn on the timeline under **Events**.

In [ ]:
from cadetgui import ProcessWidget

process.cycle_time = 900
process.add_event("load", "flow_sheet.inlet.c", 1.0, 0)
process.add_event("wash", "flow_sheet.inlet.c", 0.0, 60)
ProcessWidget(process)

## 5. Simulation

**Run simulation** runs CADET; `run()` does the same in code. The dashed line marks the retention time $t_R = \varepsilon_t V / Q$.

In [ ]:
import numpy as np

from cadetgui import SimulationWidget

t_R = 0.4 * np.pi * (0.02 / 2) ** 2 * 0.1 / 1e-7
simulation = SimulationWidget(process, markers={"t_R": t_R})
simulation.run()
simulation